# 00 — Environment check

Run this first on every machine (laptop and LinuxONE). The output doubles as **deployment evidence** for the judges:
architecture (`s390x` on LinuxONE), byte order (big-endian on IBM Z), library versions and CPU features.

In [1]:
import importlib
import os
import platform
import sys
from pathlib import Path

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"Architecture : {platform.machine()}")
print(f"Byte order   : {sys.byteorder}")
print(f"CPU count    : {os.cpu_count()}")
print(f"On IBM Z     : {platform.machine() == 's390x'}")

Python       : 3.14.4
Platform     : Linux-6.17.0-1033-oem-x86_64-with-glibc2.43
Architecture : x86_64
Byte order   : little
CPU count    : 14
On IBM Z     : False


Versions are compared against the official LinuxONE Community Cloud Jupyter Lab ML container
(`jupyterlab-image-s390x`). The scikit-learn version matters most: saved `.joblib` models only load reliably with the
version that wrote them.

In [2]:
# name -> version in the official LinuxONE container (None = not pinned)
REFERENCE = {"numpy": None, "pandas": "3.0.6", "sklearn": "1.9.0", "xgboost": "3.4.1", "networkx": "3.7",
             "matplotlib": None, "joblib": None, "requests": None, "llama_cpp": "0.3.36"}

missing = []
for name, ref in REFERENCE.items():
    try:
        version = getattr(importlib.import_module(name), "__version__", "?")
    except ImportError:
        missing.append(name)
        print(f"{name:12s} MISSING")
        continue
    note = "" if ref is None else ("  (matches container)" if version == ref else f"  (container has {ref})")
    print(f"{name:12s} {version}{note}")

if missing:
    print("\nMissing:", ", ".join(missing), "— llama_cpp is optional; without it notebook 05 uses a template explanation.")

numpy        2.5.4
pandas       3.0.6  (matches container)
sklearn      1.9.0  (matches container)
xgboost      3.4.1  (matches container)
networkx     3.7  (matches container)
matplotlib   3.11.2
joblib       1.6.0
requests     2.34.2
llama_cpp    0.3.36  (matches container)


### Platform evidence for the judges
* **Container:** the official LinuxONE setup runs Jupyter inside a Docker container.
* **zDNN:** the container's llama.cpp is built with the zDNN backend, which runs Granite's matrix operations on the
  NNPA on-chip AI accelerator.

In [3]:
in_container = (Path("/.dockerenv").exists() or Path("/run/.containerenv").exists()
                or any(k in Path("/proc/1/cgroup").read_text() for k in ("docker", "containerd", "libpod"))
                if Path("/proc/1/cgroup").exists() else False)
print("Running inside a container:", in_container)

try:
    import llama_cpp
    zdnn_libs = sorted((Path(llama_cpp.__file__).parent / "lib").glob("libggml-zdnn*.so"))
    print("llama.cpp zDNN backend   :", ", ".join(p.name for p in zdnn_libs) or "not found (CPU-only build)")
except ImportError:
    print("llama.cpp zDNN backend   : llama_cpp not installed")

Running inside a container: False
llama.cpp zDNN backend   : not found (CPU-only build)


On IBM Z, `/proc/cpuinfo` lists CPU facilities. `vx` is the vector facility; `nnpa` is the
Neural Network Processing Assist (the on-chip AI accelerator on z16 / LinuxONE 4 and newer).
Only claim accelerator use in the pitch if you actually route inference through it.

In [4]:
cpuinfo = Path("/proc/cpuinfo")
if cpuinfo.exists():
    features = next((l.split(":", 1)[1].split() for l in cpuinfo.read_text().splitlines()
                     if l.startswith("features")), [])
    if features:
        print("CPU features:", " ".join(features))
        print("Vector facility (vx):", "vx" in features)
        print("AI accelerator (nnpa):", "nnpa" in features)
    else:
        print("No 'features' line (expected on x86/ARM; present on s390x).")

No 'features' line (expected on x86/ARM; present on s390x).


### Project readiness

Checks the things that most often break on LinuxONE:
- **Writable folders:** the repo is usually cloned by the VM user (`linux1`), but Jupyter runs as the container user.
  If any check fails, run `chmod -R a+rwX ~/shared/ReliefZ` on the VM.
- **Cached road networks**, so no internet is needed.
- **Flask**, for the web UI.
- **The container's Granite model** and its byte order.
- **Memory and disk.** Granite needs a few GB of RAM. Don't load it in notebook 05 and the web UI at the same time on a
  small VM: shut down notebook 05's kernel first.

In [ ]:
import shutil
from importlib.metadata import version as package_version

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ok = lambda flag: "OK  " if flag else "FAIL"

# Every folder the notebooks and web UI write to must be writable by this (container) user
for folder in ["data", "artifacts", "logs"]:
    path = ROOT / folder
    try:
        path.mkdir(exist_ok=True)
        probe = path / ".write_test"
        probe.write_text("ok")
        probe.unlink()
        writable = True
    except OSError:
        writable = False
    print(f"{ok(writable)} {folder}/ is writable")

for graph in ["road_graph.json", "road_graph_noto.json"]:
    print(f"{ok((ROOT / 'data' / graph).exists())} data/{graph} cached (no internet needed)")
print(f"{ok((ROOT / 'webui' / 'frontend' / 'dist' / 'index.html').exists())} web UI frontend is built")
trained = all((ROOT / "artifacts" / f).exists() for f in ["model1_rf.joblib", "model2_rf.joblib", "feature_spec.json"])
print(f"{'OK  ' if trained else 'TODO'} trained models in artifacts/ (run notebooks 01-03 on this machine)")

try:
    print(f"OK   Flask {package_version('flask')} (web UI)")
except ImportError:
    print("TODO Flask not installed: pip install -r webui/requirements.txt (pure Python; works on s390x)")

try:
    from loan_demo.rag_utils import MODEL_PATH   # Granite model preinstalled in the LinuxONE container
    model = Path(MODEL_PATH)
    if model.is_file():
        with open(model, "rb") as f:
            magic, version = f.read(4), f.read(4)
        order = "little" if int.from_bytes(version, "little") < 65536 else "big"
        print(f"{ok(magic == b'GGUF' and order == sys.byteorder)} container Granite model {model} "
              f"({model.stat().st_size / 1e9:.1f} GB, {order}-endian; this machine is {sys.byteorder}-endian)")
    else:
        print(f"FAIL container Granite model path {model} does not exist")
except Exception:
    print("INFO no container Granite model (expected off LinuxONE); notebook 05 searches models/ instead")

try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"INFO memory: {mem.total / 1e9:.1f} GB total, {mem.available / 1e9:.1f} GB available")
except ImportError:
    pass
disk = shutil.disk_usage(ROOT)
print(f"INFO disk: {disk.free / 1e9:.1f} GB free")